In [1]:
import pandas as pd
from transformers import pipeline
import os
import json

c:\Users\annmarle\Desktop\Redditi postituste automaatne klassifitseerimine erineva signaalitasemega keskkondades\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
classifier = pipeline("zero-shot-classification", model = "facebook/bart-large-mnli")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 726.02it/s, Materializing param=model.shared.weight]                                   


In [ ]:
#labels = ["asking for help, advice, recommendations, or comparison","expressing an opinion, complaint, praise, or personal experience","spam, advertisement, irrelevant, or other content"]
#labels = ["question or advice-seeking post","opinion or personal experience post","spam or irrelevant post"]
#labels = [
#    "asking for help, advice, recommendations, or comparison",
#    "expressing an opinion, complaint, praise, or personal experience"
#]
labels = [
    "a post asking for advice or recommendations",
    "a post asking to compare alternatives",
    "a post sharing a personal experience or opinion",
    "a post expressing praise or complaint"
]


def classify_post(path):
    os.makedirs("../data/labeled", exist_ok=True)
    df = pd.read_json(path)
    df["title_original"] = df["title_original"].fillna("")
    df["selftext_original"] = df["selftext_original"].fillna("")

    for i, row in df.iterrows():
        text = f"{row['title_original']} {row['selftext_original']}".strip()
        result = classifier(text, labels, hypothesis_template = "The author's main intent is {}.")
        
        confidence = result["scores"][0]
        df.at[i, "confidence"] = confidence

        if result["scores"][0] < 0.6:
            df.at[i, "label"] = "UNCERTAIN: " + result["labels"][0]
        else:
            df.at[i, "label"] = result["labels"][0]
        
    print(df["label"].value_counts())
    with open(f"../data/labeled/labeled_{os.path.basename(path)}", "w", encoding="utf-8") as f:
        json.dump(df.to_dict(orient="records"), f,ensure_ascii=False,indent=4)

In [5]:
classify_post("../data/processed/clean_high.json")
classify_post("../data/processed/clean_low.json")

label
asking for help, advice, recommendations, or comparison                        581
expressing an opinion, complaint, praise, or personal experience               186
UNCERTAIN: asking for help, advice, recommendations, or comparison             112
UNCERTAIN: expressing an opinion, complaint, praise, or personal experience    109
Name: count, dtype: int64
label
asking for help, advice, recommendations, or comparison                        166
expressing an opinion, complaint, praise, or personal experience                62
UNCERTAIN: asking for help, advice, recommendations, or comparison              36
UNCERTAIN: expressing an opinion, complaint, praise, or personal experience     22
Name: count, dtype: int64
